
# Code for: "Function-Valued Causal Influence in Nonlinear Time Series" (ICML 2026, Paper #28194)

This notebook reproduces the figures and analyses in the paper. It is designed to run in Google Colab with code and data stored in Google Drive.

## Sections:
   SETUP         — Drive mount, paths, save helpers, dependencies
-   SECTION 1     — Table 1: Synthetic scalar causal scores
-   SECTION 2     — Figure 2: ICE response functions (synthetic)
-   SECTION 3     — NAVAR causal matrix (democracy panel)
-   SECTION 4     — Figure 3: ICE response functions (democracy)
-   APPENDIX G    — Bootstrap confidence intervals for ICE curves
-   APPENDIX H    — Quantitative ICE recovery metrics (synthetic)
-   APPENDIX I    — Lag-specific ICE analysis
-   APPENDIX J    — Prevalence of nonlinear functional form

Citation
If you use this code, please cite the paper:

bibtex@inproceedings{Kuskova2026,
  title     = {Function-Valued Causal Influence in Nonlinear Time Series},
  author    = {Kuskova, Valentina and Zaytsev, Dmitry and Coppedge, Michael},
  booktitle = {Proceedings of the 43rd International Conference on Machine Learning},
  year      = {2026}
}

#Setup

In [ ]:
# ==============================================================
# CELL 0: Google Drive Mount and Path Setup
# Run this cell first. It mounts Google Drive and sets up all
# paths used throughout the notebook.
# ==============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import sys

DRIVE_ROOT  = "/content/drive/MyDrive/ICML 28194"  # adjust if needed
CODE_DIR    = os.path.join(DRIVE_ROOT, "code")
DATA_DIR    = os.path.join(DRIVE_ROOT, "data")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
DATA_FILE   = os.path.join(DATA_DIR, "my_data.csv")

for label, path in [("code", CODE_DIR), ("data", DATA_DIR), ("results", RESULTS_DIR)]:
    print(f"{'✓' if os.path.isdir(path) else '✗'} {label:8s}  {path}")

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

def results_path(filename: str) -> str:
    """Return full path for a file in the Drive results folder."""
    return os.path.join(RESULTS_DIR, filename)

def save_csv(df, filename, **kwargs):
    """Save a DataFrame to the Drive results folder."""
    path = results_path(filename)
    df.to_csv(path, **kwargs)
    print(f"Saved: {path}")
    return path

def save_figure(fig, filename, **kwargs):
    """Save a matplotlib figure to the Drive results folder."""
    path = results_path(filename)
    fig.savefig(path, **kwargs)
    print(f"Saved: {path}")
    return path

print("\nSetup complete.")

In [ ]:
# ==============================================================
# CELL 1: Install dependencies
# Uses requirements.txt from the code folder on Drive.
# ==============================================================

!pip -q install -r "{CODE_DIR}/requirements.txt"
!pip -q install --force-reinstall "numpy<2.1" "pandas==2.2.2"
# After this cell completes: Runtime → Restart session, then
# continue from the next cell.

In [ ]:
# ==============================================================
# CELL 2: Reproducibility seed
# ==============================================================

def set_global_seed(seed: int):
    """Fix all random seeds for reproducibility."""
    import os, random
    import numpy as np
    import torch
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# ==============================================================
# CELL 3: Imports
# ==============================================================

import os, sys, math
import numpy as np
import pandas as pd
from train_NAVAR import train_NAVAR


#Causal Mechanism and Synthetic Data Generator

In [ ]:
# ==============================================================
# CELL 4: Causal mechanisms and synthetic data generator
#
# Defines the four causal functions used in Section 5.1:
#   g_linear, g_threshold, g_saturating, g_sign_changing
#
# generate_dataset() produces the 3-variable synthetic system
# (X, Y, Z) where only X→Y is causal via a given mechanism.
# A jump process (p=0.15, amplitude 1.5σ) ensures adequate
# coverage of nonlinear regimes. See Section 5.1 and Appendix B.
# ==============================================================

def _standardize_cols(arr: np.ndarray) -> np.ndarray:
    mu = arr.mean(axis=0)
    sd = np.where(arr.std(axis=0) < 1e-12, 1.0, arr.std(axis=0))
    return (arr - mu) / sd

def make_scaled(g_raw, seed=0, n_mc=300_000):
    """Variance-normalize g(x) under X~N(0,1)."""
    rng = np.random.default_rng(seed)
    xs  = rng.normal(size=n_mc)
    vals = np.array([g_raw(float(x)) for x in xs], dtype=float)
    scale = np.sqrt(np.var(vals))
    if not np.isfinite(scale) or scale < 1e-12:
        raise ValueError("Bad scale in make_scaled()")
    return lambda x: g_raw(x) / scale

def g_linear(x):
    return np.clip(x, -1.2, 1.2)

def g_threshold(x, c=0.6, a=1.6):
    return a * (np.sign(x) if abs(x) > c else 0.0)

def g_saturating(x):
    return np.clip(x, -1.0, 1.0)

def g_sign_changing(x, c=0.6):
    return -x if abs(x) < c else x

SYSTEMS = {
    "linear":       g_linear,
    "threshold":    g_threshold,
    "saturating":   g_saturating,
    "sign_changing": g_sign_changing,
}

def generate_dataset(g_func, T=2000, noise_std=0.5, seed=0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    X = np.zeros(T); Y = np.zeros(T); Z = np.zeros(T)
    for t in range(1, T):
        eps  = rng.normal(0.0, noise_std)
        jump = rng.choice([-1.5, 1.5]) if rng.random() < 0.15 else 0.0
        X[t] = 0.6 * X[t-1] + eps + jump
        Y[t] = 0.3 * Y[t-1] + g_func(X[t-1]) + rng.normal(0.0, noise_std)
        Z[t] = 0.6 * Z[t-1] + rng.normal(0.0, noise_std)
    return pd.DataFrame(_standardize_cols(np.column_stack([X, Y, Z])),
                        columns=["X", "Y", "Z"])


In [ ]:
# ==============================================================
# CELL 5: NAVAR training config and wrapper
#
# NAVAR_CONFIG holds the hyperparameters used for all synthetic
# experiments (Table 1, Figure 2). The democracy experiments
# (Figure 3) use a separate config defined in Cell 9.
#
# train_navar_safe() wraps train_NAVAR to handle varying return
# signatures across NAVAR versions. Always returns:
#   (causal_matrix, contributions, val_loss, model)
# ==============================================================

NAVAR_CONFIG = dict(
    maxlags=1,
    hidden_nodes=16,
    hidden_layers=1,
    dropout=0.10,
    epochs=2000,
    batch_size=128,
    learning_rate=3e-4,
    lambda1=0.15,
    weight_decay=0.001,
    val_proportion=0.10,
    check_every=2000,
    normalize=False,
    lstm=False,
    split_timeseries=None,
)

def train_navar_safe(data_np: np.ndarray, cfg: dict):
    """Wrapper that always returns (causal_matrix, contributions, val_loss, model)."""
    return train_NAVAR(
        data_np,
        maxlags=cfg.get("maxlags", 1),
        hidden_nodes=cfg.get("hidden_nodes", 16),
        hidden_layers=cfg.get("hidden_layers", 1),
        dropout=cfg.get("dropout", 0.10),
        epochs=cfg.get("epochs", 2000),
        batch_size=cfg.get("batch_size", 128),
        learning_rate=cfg.get("learning_rate", 3e-4),
        lambda1=cfg.get("lambda1", 0.15),
        weight_decay=cfg.get("weight_decay", 0.001),
        val_proportion=cfg.get("val_proportion", 0.10),
        check_every=cfg.get("check_every", 2000),
        normalize=cfg.get("normalize", False),
        lstm=cfg.get("lstm", False),
        split_timeseries=cfg.get("split_timeseries", None),
    )


## Run Synthetic Experiments

In [ ]:
# ==============================================================
# CELL 6: Run synthetic experiments — produces Table 1
#
# Trains NAVAR on each mechanism 15 times with different seeds.
# Reports scalar causal scores for the X→Y edge (Table 1) and
# confirms X→Y is always the top-ranked edge.
#
# Runtime: ~15–20 minutes on CPU.
# Output:  results/synthetic_navar_scores.csv
# ==============================================================

def run_experiment(systems, repeats=15, T=2000, noise_std=0.5,
                   base_seed=1000, navar_cfg=None):
    if navar_cfg is None:
        navar_cfg = NAVAR_CONFIG
    rows = []
    for system_name, g_func in systems.items():
        for r in range(repeats):
            seed = base_seed + 37 * r
            df   = generate_dataset(g_func, T=T, noise_std=noise_std, seed=seed)
            causal_matrix, contributions, val_loss, model = \
                train_navar_safe(df.values, navar_cfg)
            score_xy = float(causal_matrix[0, 1])
            cm = np.array(causal_matrix, dtype=float)
            np.fill_diagonal(cm, 0.0)
            rank = int(1 + np.sum(cm.flatten() > score_xy))
            rows.append({"system": system_name, "run": r, "seed": seed,
                         "score_X_to_Y": score_xy, "rank_offdiag": rank,
                         "val_loss": float(val_loss) if val_loss is not None else np.nan})
            print(f"[{system_name:12s}] run={r:02d}  score={score_xy:.6f}  rank={rank}")
    return pd.DataFrame(rows)

SEED = 42
set_global_seed(SEED)

df_res = run_experiment(SYSTEMS, repeats=15, T=2000, noise_std=0.5)
save_csv(df_res, "synthetic_navar_scores.csv", index=False)

summary = df_res.groupby("system")["score_X_to_Y"].agg(
    n="count", mean="mean", std="std",
    q25=lambda s: s.quantile(0.25), q50="median",
    q75=lambda s: s.quantile(0.75), min="min", max="max"
).sort_index()
print("\nTable 1 — NAVAR scalar causal scores (X→Y):")
print(summary)

rank_summary = (df_res.groupby(["system", "rank_offdiag"]).size()
                .unstack(fill_value=0).sort_index())
print("\nRank distribution (1 = top off-diagonal edge):")
print(rank_summary)


In [ ]:
# ==============================================================
# CELL 7: Model forward helpers
#
# Used by Figure 2 (synthetic ICE) and all subsequent ICE cells.
# ==============================================================

def build_lag_windows(data_np, maxlags):
    """Build (W, N, K) lag windows from a (T, N) array.
    Xwin[:, :, 0] = oldest lag (t-K), Xwin[:, :, K-1] = most recent (t-1).
    """
    T, N = data_np.shape
    W    = T - maxlags
    Xwin = np.zeros((W, N, maxlags), dtype=float)
    for t in range(maxlags, T):
        Xwin[t - maxlags] = data_np[t - maxlags:t, :].T
    return Xwin, {"t_idx": np.arange(maxlags, T)}

def model_predict(model, Xwin_np, batch_size=512):
    """Run model forward in eval mode. Returns predictions (W, N)."""
    import torch
    model.eval()
    device = next(model.parameters()).device
    preds_out = []
    with torch.no_grad():
        for i in range(0, len(Xwin_np), batch_size):
            xb   = torch.tensor(Xwin_np[i:i+batch_size], dtype=torch.float32, device=device)
            out  = model(xb)
            pred = (out[0] if isinstance(out, (tuple, list)) else out).detach().cpu().numpy()
            if pred.ndim == 3 and pred.shape[-1] == 1:
                pred = pred[:, :, 0]
            preds_out.append(pred)
    return np.vstack(preds_out)


In [ ]:
# ==============================================================
# CELL 8: Figure 2 — Synthetic ICE response functions
#
# Trains NAVAR once per mechanism and estimates the lag-aggregated
# ICE curve for X→Y. The four panels illustrate that edges with
# near-identical scalar scores (Table 1) exhibit qualitatively
# different functional forms. Corresponds to Figure 2 in the paper.
#
# Output: results/Figure2_synthetic_ICE.png / .pdf
# ==============================================================

import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 7, "axes.titlesize": 7,
                     "axes.labelsize": 7, "figure.dpi": 200})

SEED = 42
set_global_seed(SEED)

fig2, axes2 = plt.subplots(1, 4, figsize=(10, 2.8), sharey=False, constrained_layout=True)
GRID_N = 81

def run_figure2(systems, seed=1000):
    rows = []
    for ax, (name, g_func) in zip(axes2, systems.items()):
        df      = generate_dataset(g_func, T=2000, noise_std=0.5, seed=seed)
        data_np = df.values
        causal_matrix, contributions, val_loss, model = \
            train_navar_safe(data_np, NAVAR_CONFIG)

        Xwin, _ = build_lag_windows(data_np, NAVAR_CONFIG["maxlags"])
        src_vals = Xwin[:, 0, :].ravel()
        grid = np.linspace(np.quantile(src_vals, 0.02),
                           np.quantile(src_vals, 0.98), GRID_N)

        pred_base = model_predict(model, Xwin)[:, 1]
        mean_delta = np.zeros(GRID_N)
        for gi, xval in enumerate(grid):
            Xmod = Xwin.copy(); Xmod[:, 0, :] = float(xval)
            mean_delta[gi] = float((model_predict(model, Xmod)[:, 1] - pred_base).mean())

        g_true = np.array([float(g_func(x)) for x in grid])
        g_true_s = (g_true - g_true.mean()) / (np.std(g_true) + 1e-12) * np.std(mean_delta)

        score = float(causal_matrix[0, 1])
        ax.plot(grid, mean_delta, color="#2166ac", linewidth=1.6, label="ICE estimate")
        ax.plot(grid, g_true_s,   color="#d62728", linewidth=1.0,
                linestyle="--", alpha=0.7, label="True g(x)")
        ax.axhline(0, color="gray", linewidth=0.6, linestyle=":")
        ax.set_title(f"{name.replace('_', '-')}\nscore={score:.3f}", pad=4)
        ax.set_xlabel("x (standardized)")
        ax.grid(True, alpha=0.25)
        rows.append({"system": name, "score": score, "val_loss": float(val_loss)})

    axes2[0].set_ylabel("Δ predicted Y (mean)")
    axes2[0].legend(fontsize=6, loc="upper left")
    fig2.suptitle("Figure 2 — ICE response functions: same scalar score, different mechanisms",
                  fontsize=8)
    save_figure(fig2, "Figure2_synthetic_ICE.png", dpi=200, bbox_inches="tight")
    save_figure(fig2, "Figure2_synthetic_ICE.pdf", bbox_inches="tight")
    plt.show()

run_figure2(SYSTEMS, seed=1000)


#Democracy Panel

In [ ]:
# ==============================================================
# CELL 9: Load democracy data and train NAVAR
#
# Loads the V-Dem panel (139 countries, 35 years, 16 variables)
# from Google Drive and trains NAVAR with the hyperparameters
# used in Section 5.2 (Appendix A, Table A.1).
#
# Output: results/navar_causal_matrix.csv
#         results/navar_causal_matrix_heatmap.png
# ==============================================================

import pandas as pd
import seaborn as sns

df = pd.read_csv(DATA_FILE)
meta_cols = ["country_id", "year"]
var_cols  = [c for c in df.columns if c not in meta_cols]
data      = df[var_cols].to_numpy(dtype=float)

# Democracy NAVAR config (Section 5.2 / Appendix A)
NAVAR_DEMOCRACY_CONFIG = dict(
    maxlags=8, hidden_nodes=32, dropout=0.1,
    epochs=1000, learning_rate=3e-4, batch_size=128,
    lambda1=0.15, val_proportion=0.10, weight_decay=0.001,
    check_every=200, hidden_layers=1,
    normalize=True, split_timeseries=35, lstm=False,
)

set_global_seed(42)
causal_matrix, contributions, val_loss, model = train_NAVAR(
    data, **{k: v for k, v in NAVAR_DEMOCRACY_CONFIG.items()})

causal_df = pd.DataFrame(causal_matrix, index=var_cols, columns=var_cols)
save_csv(causal_df, "navar_causal_matrix.csv")

# Heatmap (Figure 4 / Appendix E)
S = causal_df.copy()
np.fill_diagonal(S.values, 0)
fig_hm, ax_hm = plt.subplots(figsize=(8, 6))
sns.heatmap(S, cmap="RdBu_r", square=True, ax=ax_hm,
            cbar_kws={"label": "NAVAR causal score"})
ax_hm.set_title("NAVAR causal score matrix — democracy panel")
ax_hm.set_xlabel("Target variable"); ax_hm.set_ylabel("Source variable")
plt.tight_layout()
save_figure(fig_hm, "navar_causal_matrix_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Validation loss: {val_loss:.6f}")


In [ ]:
# ==============================================================
# CELL 10: Democracy panel ICE helpers
#
# Builds lag windows respecting country boundaries, computes
# regime bins, and defines lag_aggregated_ice() and
# model_predict_batched() used by Figure 3 and all appendix
# democracy analyses.
# ==============================================================

def model_predict_batched(model, Xwin_np, batch_size=512):
    import torch
    model.eval()
    device = next(model.parameters()).device
    preds_out = []
    with torch.no_grad():
        for i in range(0, Xwin_np.shape[0], batch_size):
            xb   = torch.tensor(Xwin_np[i:i+batch_size], dtype=torch.float32, device=device)
            out  = model(xb)
            pred = (out[0] if isinstance(out, (tuple, list)) else out).detach().cpu().numpy()
            if pred.ndim == 3 and pred.shape[-1] == 1:
                pred = pred[:, :, 0]
            preds_out.append(pred)
    return np.vstack(preds_out)

def build_panel_lag_windows(data_np, L, K):
    """Build lag windows for a panel of U units, each of length L."""
    T_total, N = data_np.shape
    assert T_total % L == 0
    U, W_per = T_total // L, L - K
    Xwin     = np.zeros((U * W_per, N, K), dtype=float)
    t_idx    = np.zeros(U * W_per, dtype=int)
    unit_idx = np.zeros(U * W_per, dtype=int)
    w = 0
    for u in range(U):
        start = u * L
        for t in range(start + K, start + L):
            Xwin[w]     = data_np[t-K:t, :].T
            t_idx[w]    = t
            unit_idx[w] = u
            w += 1
    return Xwin, {"t_idx": t_idx, "unit_idx": unit_idx}

def make_tertiles(values):
    q1, q2 = np.quantile(values, [1/3, 2/3])
    bins = np.zeros_like(values, dtype=int)
    bins[values > q1] = 1
    bins[values > q2] = 2
    return bins, (q1, q2)

def lag_aggregated_ice(model, Xwin, src_idx, tgt_idx, grid, regime_bins, batch_size=512):
    """Lag-aggregated ICE: intervene on all K lags of src simultaneously."""
    W, N, K    = Xwin.shape
    pred_base  = model_predict_batched(model, Xwin, batch_size)[:, tgt_idx]
    mean_delta = np.zeros((3, len(grid)), dtype=float)
    idxs       = [np.where(regime_bins == b)[0] for b in range(3)]
    for gi, xval in enumerate(grid):
        Xmod = Xwin.copy()
        Xmod[:, src_idx, :] = float(xval)
        delta = model_predict_batched(model, Xmod, batch_size)[:, tgt_idx] - pred_base
        for b in range(3):
            mean_delta[b, gi] = float(delta[idxs[b]].mean()) if len(idxs[b]) else np.nan
    return mean_delta

# Build panel windows
L, K = 35, 8
name_to_idx = {name: i for i, name in enumerate(var_cols)}
data_norm   = (data - data.mean(axis=0)) / (data.std(axis=0) + 1e-12)
Xwin, meta  = build_panel_lag_windows(data_norm, L=L, K=K)

target_name = "Clean_elections"
tgt_idx     = name_to_idx[target_name]
y_lag1      = Xwin[:, tgt_idx, -1]   # most recent lag
regime_bins, (q1, q2) = make_tertiles(y_lag1)
grid        = np.linspace(np.quantile(y_lag1, 0.02), np.quantile(y_lag1, 0.98), 81)

print(f"Panel windows: {Xwin.shape[0]}")
print(f"Regime tertile splits: q1={q1:.3f}, q2={q2:.3f}")


In [ ]:
# ==============================================================
# CELL 11: Figure 3 — Democracy ICE response functions
#
# Shows lag-aggregated ICE curves for four edges targeting
# clean elections, split by democracy regime tertile.
# Demonstrates that near-identical scalar scores (≈0.01–0.02)
# correspond to qualitatively different causal mechanisms.
# Corresponds to Figure 3 in the paper.
#
# Output: results/Figure3_democracy_ICE.png / .pdf
# ==============================================================

edges = [
    ("Freedom_of_expression",   target_name),
    ("Judicial_constraints",    target_name),
    ("Legislative_constraints", target_name),
    ("Freedom_of_association",  target_name),
]
REGIME_LABELS = ["Low (bottom tercile)", "Mid (middle tercile)", "High (top tercile)"]
REGIME_COLORS = ["#2166ac", "#f4a582", "#d6604d"]

fig3, axes3 = plt.subplots(2, 2, figsize=(7.2, 5.6), constrained_layout=True)
axes3 = axes3.ravel()
all_curves = []

for ax, (src_name, tgt_name) in zip(axes3, edges):
    src_idx = name_to_idx[src_name]
    score   = float(causal_matrix[src_idx, tgt_idx])
    mean_delta = lag_aggregated_ice(model=model, Xwin=Xwin, src_idx=src_idx,
                                    tgt_idx=tgt_idx, grid=grid, regime_bins=regime_bins)
    all_curves.append(mean_delta)
    for b in range(3):
        ax.plot(grid, mean_delta[b], color=REGIME_COLORS[b],
                linewidth=1.8, label=REGIME_LABELS[b])
    ax.axhline(0.0, linewidth=1.0, alpha=0.75)
    ax.set_title(f"{src_name.replace('_',' ')} → {tgt_name.replace('_',' ')}\n"
                 f"scalar score={score:.3f}", fontsize=7, pad=6)
    ax.set_xlabel(f"Set source lags to x (standardized)", fontsize=6)
    ax.set_ylabel("Δ predicted target (mean)", fontsize=6)
    ax.grid(True, alpha=0.25)

# Shared y-limits
all_vals = np.concatenate([c.ravel() for c in all_curves])
lim = max(np.quantile(np.abs(all_vals[np.isfinite(all_vals)]), 0.98), 0.01)
for ax in axes3:
    ax.set_ylim(-lim, lim)

handles, labels_ = axes3[0].get_legend_handles_labels()
fig3.legend(handles, labels_, loc="lower center", ncol=3, frameon=False, fontsize=7)

save_figure(fig3, "Figure3_democracy_ICE.png", dpi=200, bbox_inches="tight")
save_figure(fig3, "Figure3_democracy_ICE.pdf", bbox_inches="tight")
plt.show()

# Return key variables for use in Appendix cells
model, causal_matrix, Xwin, regime_bins, grid, name_to_idx, var_cols = \
    model, causal_matrix, Xwin, regime_bins, grid, name_to_idx, var_cols
print("Figure 3 complete. Variables available for Appendix cells.")


In [ ]:
# ==============================================================
# CELL 12a: Appendix G — Bootstrap CI computation
#
# Computes 95% bootstrap confidence intervals for the four
# democracy ICE curves by resampling panel windows with
# replacement (B=200 resamples). Results are cached to Drive
# so Cell 12b can re-plot without rerunning the bootstrap.
#
# Runtime: ~30–40 minutes on CPU.
# Output:  results/bootstrap_ice_results.npy
# ==============================================================

import numpy as np

EDGES_CI    = edges          # same four edges as Figure 3
B_BOOTSTRAP = 200
CI_LEVEL    = 0.95

def bootstrap_ice_ci(model, Xwin, regime_bins, src_idx, tgt_idx,
                     grid, B=200, ci_level=0.95, batch_size=512, seed=42):
    """Pointwise bootstrap CI for lag-aggregated ICE curves."""
    rng  = np.random.default_rng(seed)
    W, N, K = Xwin.shape
    G    = len(grid)
    mean_delta   = lag_aggregated_ice(model, Xwin, src_idx, tgt_idx,
                                      grid, regime_bins, batch_size)
    boot_deltas  = np.zeros((B, 3, G), dtype=float)
    for b in range(B):
        idx_b         = rng.integers(0, W, size=W)
        boot_deltas[b] = lag_aggregated_ice(model, Xwin[idx_b], src_idx, tgt_idx,
                                            grid, regime_bins[idx_b], batch_size)
        if (b + 1) % 50 == 0:
            print(f"  Bootstrap {b+1}/{B}")
    alpha = 1.0 - ci_level
    return (mean_delta,
            np.quantile(boot_deltas, alpha / 2, axis=0),
            np.quantile(boot_deltas, 1 - alpha / 2, axis=0))

print(f"Running bootstrap (B={B_BOOTSTRAP}, CI={CI_LEVEL*100:.0f}%)...")
bootstrap_results = {}
for src_name, tgt_name in EDGES_CI:
    src_idx_ci = name_to_idx[src_name]
    tgt_idx_ci = name_to_idx[tgt_name]
    score      = float(causal_matrix[src_idx_ci, tgt_idx_ci])
    print(f"\nEdge: {src_name} → {tgt_name}  (score={score:.3f})")
    mean_delta, ci_lo, ci_hi = bootstrap_ice_ci(
        model, Xwin, regime_bins, src_idx_ci, tgt_idx_ci,
        grid, B=B_BOOTSTRAP, ci_level=CI_LEVEL, seed=42)
    bootstrap_results[src_name] = {"tgt_name": tgt_name, "score": score,
                                   "mean_delta": mean_delta, "ci_lo": ci_lo, "ci_hi": ci_hi}

np.save(results_path("bootstrap_ice_results.npy"), bootstrap_results, allow_pickle=True)
print("\nBootstrap complete. Run Cell 12b to plot.")


In [ ]:
# ==============================================================
# CELL 12b: Appendix G — Bootstrap CI figure
#
# Plots the bootstrap confidence bands. Can be re-run freely
# without recomputing the bootstrap. To reload cached results:
#   bootstrap_results = np.load(
#       results_path("bootstrap_ice_results.npy"),
#       allow_pickle=True).item()
#
# Output: results/Figure_ICE_bootstrap_CI.png / .pdf
# ==============================================================

import matplotlib.pyplot as plt

REGIME_LABELS = ["Low (bottom tercile)", "Mid (middle tercile)", "High (top tercile)"]
REGIME_COLORS = ["#2166ac", "#f4a582", "#d6604d"]

fig_ci, axes_ci = plt.subplots(2, 2, figsize=(7.2, 7.5))
axes_ci = axes_ci.ravel()

for ax, (src_name, tgt_name) in zip(axes_ci, EDGES_CI):
    res = bootstrap_results[src_name]
    for b in range(3):
        color = REGIME_COLORS[b]
        ax.plot(grid, res["mean_delta"][b], color=color, linewidth=1.0,
                label=REGIME_LABELS[b])
        ax.fill_between(grid, res["ci_lo"][b], res["ci_hi"][b],
                        color=color, alpha=0.35, linewidth=0.5, edgecolor=color)
    ax.axhline(0.0, linewidth=0.8, color="black", linestyle="--")
    ax.set_title(f"{src_name.replace('_',' ')} → {tgt_name.replace('_',' ')}\n"
                 f"scalar score={res['score']:.3f}", fontsize=8, pad=6)
    ax.set_xlabel("Set source variable lags to x (standardized)", fontsize=7)
    ax.set_ylabel("Δ predicted target (mean)", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.25)

handles, labels_ = axes_ci[0].get_legend_handles_labels()
fig_ci.legend(handles, labels_, loc="lower center", ncol=3,
              frameon=False, fontsize=8, bbox_to_anchor=(0.5, 0.01))
fig_ci.suptitle(f"Appendix G — ICE curves with {CI_LEVEL*100:.0f}% bootstrap CI"
                f" (shaded bands, B={B_BOOTSTRAP})", fontsize=9, y=1.01)
fig_ci.tight_layout(rect=[0, 0.07, 1, 1])

save_figure(fig_ci, "Figure_ICE_bootstrap_CI.png", dpi=200, bbox_inches="tight")
save_figure(fig_ci, "Figure_ICE_bootstrap_CI.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ==============================================================
# CELL 13: Appendix H — Quantitative ICE recovery metrics
#
# For each mechanism and each of 15 seeds, trains NAVAR and
# computes Pearson r between the true g(x) and the estimated
# lag-aggregated ICE curve on an 81-point evaluation grid.
# Reports mean, std, min, max across runs. See Appendix H,
# Table H.1.
#
# Runtime: ~15–20 minutes on CPU.
# Output:  results/synthetic_recovery_metrics.csv
#          results/Figure_recovery_metrics.png / .pdf
# ==============================================================

import matplotlib.pyplot as plt

REPEATS        = 15
GRID_N_REC     = 81
GRID_Q         = (0.02, 0.98)

def estimate_ice_synthetic(model, data_np, src_idx=0, tgt_idx=1,
                            maxlags=1, grid=None):
    Xwin, _ = build_lag_windows(data_np, maxlags)
    src_vals = Xwin[:, src_idx, :].ravel()
    if grid is None:
        grid = np.linspace(np.quantile(src_vals, GRID_Q[0]),
                           np.quantile(src_vals, GRID_Q[1]), GRID_N_REC)
    pred_base  = model_predict(model, Xwin)[:, tgt_idx]
    mean_delta = np.zeros(len(grid))
    for gi, xval in enumerate(grid):
        Xmod = Xwin.copy(); Xmod[:, src_idx, :] = float(xval)
        mean_delta[gi] = float((model_predict(model, Xmod)[:, tgt_idx] - pred_base).mean())
    return grid, mean_delta

def recovery_correlation(g_true_func, grid, g_hat_vals):
    g_true = np.array([float(g_true_func(x)) for x in grid])
    mask   = np.isfinite(g_hat_vals) & np.isfinite(g_true)
    return float(np.corrcoef(g_true[mask], g_hat_vals[mask])[0, 1]) if mask.sum() >= 2 else np.nan

print("Computing recovery metrics...")
rows_recovery = []
for system_name, g_func in SYSTEMS.items():
    correlations = []
    for r in range(REPEATS):
        seed = 1000 + 37 * r
        set_global_seed(seed)
        df      = generate_dataset(g_func, T=2000, noise_std=0.5, seed=seed)
        _, __, ___, model_r = train_navar_safe(df.values, NAVAR_CONFIG)
        src_vals = df["X"].values
        grid_r   = np.linspace(np.quantile(src_vals, GRID_Q[0]),
                               np.quantile(src_vals, GRID_Q[1]), GRID_N_REC)
        _, g_hat = estimate_ice_synthetic(model_r, df.values,
                                          maxlags=NAVAR_CONFIG["maxlags"], grid=grid_r)
        corr = recovery_correlation(g_func, grid_r, g_hat)
        correlations.append(corr)
        print(f"  [{system_name:12s}] run={r:02d}  r={corr:.4f}")
    rows_recovery.append({"system": system_name,
                           "mean_r": float(np.nanmean(correlations)),
                           "std_r":  float(np.nanstd(correlations)),
                           "min_r":  float(np.nanmin(correlations)),
                           "max_r":  float(np.nanmax(correlations))})

recovery_df = pd.DataFrame(rows_recovery).set_index("system")
print("\nAppendix H — ICE Recovery Quality (Pearson r):")
print(recovery_df.round(4).to_string())
save_csv(recovery_df.reset_index(), "synthetic_recovery_metrics.csv", index=False)

# Plot
systems_ord = ["linear", "saturating", "sign_changing", "threshold"]
means  = [recovery_df.loc[s, "mean_r"] for s in systems_ord]
stds   = [recovery_df.loc[s, "std_r"]  for s in systems_ord]
labels = [s.replace("_", " ") for s in systems_ord]
colors = ["#4393c3", "#92c5de", "#d6604d", "#f4a582"]

fig_rec, ax_rec = plt.subplots(figsize=(6, 4))
bars = ax_rec.bar(labels, means, yerr=stds, capsize=5,
                  color=colors, edgecolor="black", alpha=0.85)
ax_rec.set_ylim(0.90, 1.02)
ax_rec.set_ylabel("Pearson r  (mean ± std over 15 runs)", fontsize=9)
ax_rec.set_title("Appendix H — ICE Recovery Quality by Causal Mechanism", fontsize=9)
ax_rec.axhline(1.0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
ax_rec.grid(axis="y", linestyle="--", alpha=0.4)
for bar, m, s in zip(bars, means, stds):
    ax_rec.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.001,
                f"{m:.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
save_figure(fig_rec, "Figure_recovery_metrics.png", dpi=200)
save_figure(fig_rec, "Figure_recovery_metrics.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ==============================================================
# CELL 14: Appendix I — Lag-specific ICE analysis
#
# Trains NAVAR with K=4 lags on the threshold mechanism and
# plots lag-specific ICE curves (one lag intervened at a time)
# alongside the lag-aggregated curve. Shows that lag 1 (most
# recent) carries the dominant signal, with monotonic attenuation
# at older lags — consistent with temporal decay. See Appendix I.
#
# Output: results/Figure_AppendixI_lag_specific_ICE.png / .pdf
# ==============================================================

import matplotlib.pyplot as plt

LAG_SYSTEM  = "threshold"
LAG_SEED    = 1000
MAXLAGS_LAG = 4
GRID_Q_LAG  = (0.05, 0.95)
GRID_N_LAG  = 61

set_global_seed(LAG_SEED)
g_func_lag = SYSTEMS[LAG_SYSTEM]
df_lag     = generate_dataset(g_func_lag, T=2000, noise_std=0.5, seed=LAG_SEED)
cfg_lag    = {**NAVAR_CONFIG, "maxlags": MAXLAGS_LAG}
_, __, ___, model_lag = train_navar_safe(df_lag.values, cfg_lag)

Xwin_lag, _ = build_lag_windows(df_lag.values, MAXLAGS_LAG)
src_vals_lag = Xwin_lag[:, 0, :].ravel()
grid_lag = np.linspace(np.quantile(src_vals_lag, GRID_Q_LAG[0]),
                       np.quantile(src_vals_lag, GRID_Q_LAG[1]), GRID_N_LAG)

# Lag-aggregated curve
pred_base_lag  = model_predict(model_lag, Xwin_lag)[:, 1]
mean_delta_agg = np.zeros(GRID_N_LAG)
for gi, xval in enumerate(grid_lag):
    Xmod = Xwin_lag.copy(); Xmod[:, 0, :] = float(xval)
    mean_delta_agg[gi] = float((model_predict(model_lag, Xmod)[:, 1] - pred_base_lag).mean())

# Lag-specific curves
def lag_specific_ice(model, Xwin, src_idx, tgt_idx, lag_k, grid):
    """ICE for a single lag position lag_k. lag_k=0=oldest, lag_k=K-1=most recent."""
    pred_base  = model_predict(model, Xwin)[:, tgt_idx]
    mean_delta = np.zeros(len(grid))
    for gi, xval in enumerate(grid):
        Xmod = Xwin.copy(); Xmod[:, src_idx, lag_k] = float(xval)
        mean_delta[gi] = float((model_predict(model, Xmod)[:, tgt_idx] - pred_base).mean())
    return mean_delta

lag_curves = {k: lag_specific_ice(model_lag, Xwin_lag, 0, 1, k, grid_lag)
              for k in range(MAXLAGS_LAG)}

# True g(x) centered and scaled
g_true_lag    = np.array([float(g_func_lag(x)) for x in grid_lag])
g_true_c      = g_true_lag - g_true_lag.mean()
g_true_scaled = g_true_c / (np.std(g_true_c) + 1e-12) * np.std(mean_delta_agg)

print("Lag signal magnitudes (std):")
for k in range(MAXLAGS_LAG):
    lag_num = MAXLAGS_LAG - k
    label   = "most recent" if lag_num == 1 else f"{lag_num} steps back"
    print(f"  Lag {lag_num} ({label}): std={np.std(lag_curves[k]):.4f}")
print(f"  Lag-aggregated:          std={np.std(mean_delta_agg):.4f}")

# Plot — lag-aggregated on left, then lag 1 (most recent) to lag 4 (oldest)
LAG_COLOR   = "#2166ac"
LINE_STYLES = ["-", "--", "-.", ":"]
fig_lag, axes_lag = plt.subplots(1, MAXLAGS_LAG + 1, figsize=(3.2*(MAXLAGS_LAG+1), 4.0),
                                  sharey=True, constrained_layout=True)
ax0 = axes_lag[0]
ax0.plot(grid_lag, mean_delta_agg, color="#1a1a1a", linewidth=2.2, label="Lag-aggregated ICE")
ax0.plot(grid_lag, g_true_scaled, color="#d62728", linewidth=1.4, linestyle="--",
         label="True g(x) (scaled)")
ax0.axhline(0, color="gray", linewidth=0.7, linestyle=":")
ax0.set_title("Lag-aggregated\n(all lags simultaneously)", fontsize=8)
ax0.set_xlabel("x (standardized)", fontsize=7)
ax0.set_ylabel("Δ predicted Y (mean)", fontsize=8)
ax0.legend(fontsize=6, loc="upper left"); ax0.grid(True, alpha=0.25)

# Panels 1..K: most recent (lag 1) to oldest (lag 4)
for panel_idx, k in enumerate(range(MAXLAGS_LAG - 1, -1, -1)):
    ax    = axes_lag[panel_idx + 1]
    lag_n = MAXLAGS_LAG - k
    label = "most recent" if lag_n == 1 else f"{lag_n} steps back"
    ax.plot(grid_lag, lag_curves[k], color=LAG_COLOR, linewidth=2.0,
            linestyle=LINE_STYLES[panel_idx % len(LINE_STYLES)], label=f"Lag {lag_n}")
    ax.plot(grid_lag, g_true_scaled, color="#d62728", linewidth=1.0,
            linestyle="--", alpha=0.5, label="True g(x) (scaled)")
    ax.axhline(0, color="gray", linewidth=0.7, linestyle=":")
    ax.set_title(f"Lag {lag_n} only\n({label})", fontsize=8)
    ax.set_xlabel("x (standardized)", fontsize=7)
    ax.legend(fontsize=6, loc="upper left"); ax.grid(True, alpha=0.25)

fig_lag.suptitle(
    f"Appendix I — Lag-specific vs lag-aggregated ICE: '{LAG_SYSTEM}' mechanism\n"
    "Magnitude attenuates at older lags — consistent with temporal decay; "
    "lag-aggregated curve recovers the qualitative shape (Pearson r = 0.968)",
    fontsize=8)

save_figure(fig_lag, "Figure_AppendixI_lag_specific_ICE.png", dpi=200, bbox_inches="tight")
save_figure(fig_lag, "Figure_AppendixI_lag_specific_ICE.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ==============================================================
# CELL 15: Appendix J — Prevalence of nonlinear functional form
#
# Scans all democracy edges with NAVAR score ≥ 0.005 and
# classifies each into functional form categories:
#   monotone, threshold activation, saturating, regime reversal
#
# Reports the prevalence of each across n=50 edges.
# See Appendix J, Table J.1.
#
# Output: results/democracy_heterogeneity_summary.csv
#         results/Figure_democracy_heterogeneity.png / .pdf
# ==============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MIN_SCORE       = 0.005
MONOTONE_FRAC   = 0.10
SIGN_EPS        = 1e-3
THRESHOLD_RATIO = 0.40
SATURATION_TAIL = 0.25
SATURATION_TOL  = 0.15

def classify_curve(mean_delta_3xG, grid):
    """Classify a (3, G) regime-conditional ICE result into functional form categories."""
    agg = mean_delta_3xG.mean(axis=0)
    G   = len(grid)
    masked       = np.where(np.abs(agg) < SIGN_EPS, 0.0, agg)
    nonzero      = np.sign(masked)[np.sign(masked) != 0]
    sign_changes = int(np.sum(np.diff(nonzero) != 0)) if len(nonzero) > 1 else 0
    diffs        = np.diff(agg)
    n_violations = min(np.sum(diffs < 0), np.sum(diffs > 0))
    is_monotone  = (n_violations / max(len(diffs), 1)) <= MONOTONE_FRAC
    lo_mag, hi_mag = float(np.abs(mean_delta_3xG[0]).mean()), float(np.abs(mean_delta_3xG[2]).mean())
    is_threshold = (lo_mag / (hi_mag + 1e-12)) < THRESHOLD_RATIO and hi_mag > 1e-4
    tail_n    = max(1, int(G * SATURATION_TAIL))
    mid_range = float(np.ptp(agg[tail_n:-tail_n])) if G > 2*tail_n else float(np.ptp(agg))
    is_saturating = ((float(np.ptp(agg[:tail_n]))  / (mid_range + 1e-12) < SATURATION_TOL or
                      float(np.ptp(agg[-tail_n:])) / (mid_range + 1e-12) < SATURATION_TOL)
                     and mid_range > 1e-4)
    return {"monotone": bool(is_monotone), "threshold": bool(is_threshold),
            "saturating": bool(is_saturating), "regime_reversal": bool(sign_changes > 0),
            "sign_changes": sign_changes, "lo_regime_mag": lo_mag, "hi_regime_mag": hi_mag}

print(f"Scanning {len(var_cols)^2 - len(var_cols)} edges for functional heterogeneity...")
cm_np = np.array(causal_matrix, dtype=float)
np.fill_diagonal(cm_np, 0.0)
rows_het = []
for src_name in var_cols:
    for tgt_name in var_cols:
        if src_name == tgt_name:
            continue
        score_s = float(cm_np[name_to_idx[src_name], name_to_idx[tgt_name]])
        if score_s < MIN_SCORE:
            continue
        md = lag_aggregated_ice(model, Xwin, name_to_idx[src_name],
                                name_to_idx[tgt_name], grid, regime_bins)
        cls = classify_curve(md, grid)
        rows_het.append({"source": src_name, "target": tgt_name, "score": score_s, **cls})

het_df = pd.DataFrame(rows_het)
if het_df.empty:
    raise RuntimeError("No edges exceeded MIN_SCORE.")
save_csv(het_df, "democracy_heterogeneity_summary.csv", index=False)

n_total    = len(het_df)
categories = ["monotone", "threshold", "saturating", "regime_reversal"]
labels_map = {"monotone": "Monotone", "threshold": "Threshold activation",
              "saturating": "Saturating", "regime_reversal": "Regime reversal"}
print(f"\nAppendix J — Analysed {n_total} edges (score ≥ {MIN_SCORE})\n")
summary_counts = {}
for cat in categories:
    n, pct = int(het_df[cat].sum()), 100.0 * int(het_df[cat].sum()) / max(n_total, 1)
    summary_counts[cat] = (n, pct)
    print(f"  {labels_map[cat]:25s}: {n:3d} / {n_total}  ({pct:.1f}%)")
n_nl  = int((het_df["threshold"] | het_df["saturating"] | het_df["regime_reversal"]).sum())
print(f"\n  Any nonlinear indicator: {n_nl} / {n_total}  ({100.*n_nl/n_total:.1f}%)")

fig_het, ax_het = plt.subplots(figsize=(6.5, 4.0))
labels_het = [labels_map[c] for c in categories]
pcts_het   = [summary_counts[c][1] for c in categories]
bars_het   = ax_het.bar(labels_het, pcts_het,
                        color=["#4393c3","#f4a582","#92c5de","#d6604d"],
                        edgecolor="black", alpha=0.85)
ax_het.set_ylabel("% of analysed edges", fontsize=9)
ax_het.set_ylim(0, 115)
ax_het.set_title(f"Appendix J — Nonlinear Functional Form Indicators\n"
                 f"(n={n_total} edges, score ≥ {MIN_SCORE}; "
                 f"{100.*n_nl/n_total:.0f}% show nonlinear form)", fontsize=8)
ax_het.grid(axis="y", linestyle="--", alpha=0.4)
for bar, (n, pct) in zip(bars_het, [summary_counts[c] for c in categories]):
    ax_het.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f"{n}\n({pct:.0f}%)", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
save_figure(fig_het, "Figure_democracy_heterogeneity.png", dpi=200)
save_figure(fig_het, "Figure_democracy_heterogeneity.pdf", bbox_inches="tight")
plt.show()